# Data Preparation Pipeline


## Introduction

This notebook applies a sequence of feature engineering and data transformation steps defined in external configuration files.

Each configuration file represents a set of decisions made during earlier stages of analysis (EDA, feature engineering, and feature selection). These decisions are stored and then reapplied in a structured and reproducible way.

By separating transformation logic from execution, this approach improves transparency, reduces manual errors, and better reflects how production data pipelines are designed.

## Table of Contents

1. [Loading Data](#loading-data)
2. [Loading Transformation Configurations](#loading-transformation-configurations)
3. [Applying Data Transformation Pipeline](#applying-data-transformation-pipeline)
4. [Sanity Checks and Validation](#sanity-checks-and-validation)
5. [Applying Data Transformation Pipeline on Test Data](#applying-data-transformation-pipeline-test-data)
6. [Conclusion](#conclusion)


## Loading Data

In this step, the training and test datasets are loaded from the processed data directory.

These datasets were previously created during the train-test split step and serve as the input for the data preparation pipeline.

The training dataset will be used to apply all transformation steps, while the test dataset will be processed using the same pipeline to ensure consistency between model training and evaluation.

In [1]:
import polars as pl

# Load train and test datasets
train_df = pl.read_csv("./data/processed/02_train_dataset.csv")
test_df = pl.read_csv("./data/processed/02_test_dataset.csv")

# Quick check
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (5634, 21)
Test shape: (1409, 21)


## Loading Transformation Configurations

In this step, transformation configurations (artifacts) are loaded from external JSON files.

Each configuration file represents a set of decisions made during earlier stages of the project, including:
- feature releveling and encoding,
- feature aggregation,
- binning strategies,
- and feature selection (dropping columns).

By storing these decisions outside of the notebook, we separate **analytical reasoning** from **execution logic**. This improves:
- reproducibility,
- transparency of transformations,
- and consistency across training and test data.

These configurations will be applied sequentially to reconstruct the final model-ready dataset.

In [2]:
# load artifacts

import json

with open("./data/configs/01_eda.json") as f:
    cfg_01 = json.load(f)

with open("./data/configs/02_ram.json") as f:
    cfg_02 = json.load(f)

with open("./data/configs/03_fe.json") as f:
    cfg_03 = json.load(f)

with open("./data/configs/04_fe.json") as f:
    cfg_04 = json.load(f)

with open("./data/configs/05_fe.json") as f:
    cfg_05 = json.load(f)

with open("./data/configs/06_fe.json") as f:
    cfg_06 = json.load(f)

artifacts = [cfg_01, cfg_02, cfg_03, cfg_04, cfg_05, cfg_06]

## Applying Data Transformation Pipeline

The data preparation pipeline is executed by iteratively applying transformation artifacts to the dataset.

The transformation logic is encapsulated in a reusable helper function defined in an external module. This enables a clear separation between:
- configuration (what transformations should be applied),
- and implementation (how transformations are executed).

Each artifact is processed in sequence, allowing the pipeline to reconstruct the final feature set based on previously defined analytical decisions.

This design follows a config-driven approach, improving:
- reproducibility of results,
- maintainability of transformation logic,
- and consistency across different stages of the workflow.

It also reflects real-world data engineering practices, where transformation pipelines are modular and reusable rather than tightly coupled to notebook execution.

In [ ]:
from src.utils.data_preparation_pipeline import apply_artifact

df_out = train_df
for artifact in artifacts:
    df_out = apply_artifact(df_out, artifact)

## Sanity Checks and Validation

To verify the correctness of the data preparation pipeline, the generated dataset is compared against a reference dataset created during the feature engineering stage.

Before comparison, column order is aligned to ensure a fair equality check. The datasets are then compared using a strict equality check, confirming that:
- all features are present,
- values are identical,
- and data types are consistent.

This validation step ensures that the pipeline correctly reproduces the expected transformations and that the final dataset is reliable for downstream modeling.

In [4]:
fe_end_train_df = pl.read_csv("./data/processed/05_fe_train_df_end.csv")

df_out = df_out.select(sorted(df_out.columns))
fe_end_train_df = fe_end_train_df.select(sorted(fe_end_train_df.columns))

df_out.equals(fe_end_train_df)

True

## Applying Data Transformation Pipeline (Test Data)

In this step, the same transformation pipeline is applied to the test dataset.

The test data has not been used during exploratory analysis or feature engineering, and remains untouched since the initial train-test split. Applying the pipeline ensures that the test dataset undergoes the exact same transformations as the training data.

This is critical to maintain consistency between training and evaluation, as the model expects the same feature structure and preprocessing steps.

By reusing the same configuration-driven pipeline, we ensure that:
- no information from the test set influenced transformation decisions,
- preprocessing is applied consistently,
- and the resulting dataset is suitable for unbiased model evaluation.

In [5]:
df_test_out = test_df
for artifact in artifacts:
    df_test_out = apply_artifact(df_test_out, artifact)

In [6]:
print(
    sorted(df_out.columns) == sorted(df_test_out.columns)
)

True


In [7]:
df1 = df_out.select(sorted(df_out.columns))
df2 = df_test_out.select(sorted(df_test_out.columns))

print(df1.schema == df2.schema)

True


**Summary**

The transformation pipeline was successfully applied to the test dataset using the same configuration-driven approach as for the training data.

Since the test data was not used during exploratory analysis or feature engineering, it represents unseen data. Instead of validating against a reference dataset, validation focused on structural consistency.

The resulting test dataset matches the training dataset in terms of feature names and data types, confirming that the pipeline generalizes correctly and produces a model-ready dataset suitable for unbiased evaluation.

## Conclusion

The data preparation pipeline was successfully executed using the saved transformation artifacts from previous stages of the project.

After applying all transformations sequentially and aligning column order, the recreated dataset matched the expected engineered training dataset exactly. This confirms that the pipeline correctly reproduces the feature engineering logic in a structured and reusable way.

This step provides a reproducible bridge between exploratory analysis and modeling, ensuring that the final model-ready data can be generated consistently from the saved configuration artifacts.

In [8]:
df_out.write_csv("data/processed/06_dpp_train_df.csv")
df_test_out.write_csv("data/processed/06_dpp_test_df.csv")